In [1]:
!pip install pypdf
!pip install sentence-transformers
import numpy as np
import io
import re
from pypdf import PdfReader
from pyspark.sql import SparkSession
import uuid
import numpy as np
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, ArrayType, FloatType
from pyspark.sql.functions import udf, lit
from pyspark.ml.feature import Tokenizer, HashingTF, IDF



You should consider upgrading via the '/home/guest/anaconda3/bin/python -m pip install --upgrade pip' command.


You should consider upgrading via the '/home/guest/anaconda3/bin/python -m pip install --upgrade pip' command.


pourquoi spark

Le travail en parallèle (Distribué) : Au lieu d'utiliser un seul cœur de ton processeur, il utilise tous les cœurs de ta machine en même temps (voire plusieurs ordinateurs reliés entre eux dans de vraies entreprises).

Le traitement en mémoire (In-Memory) : Avant Spark, l'ancien système (Hadoop MapReduce) écrivait les résultats sur le disque dur entre chaque étape, ce qui était très lent. Spark garde un maximum de choses dans la mémoire vive (RAM). C'est jusqu'à 100 fois plus rapide !

La "Paresse" intelligente (Lazy Evaluation) : Quand tu écris ton code (comme notre flatMap), Spark ne fait rien. Il prend juste des notes et prépare "le meilleur plan d'attaque". Il ne lance réellement les calculs que quand tu lui demandes un résultat final (par exemple avec la commande .take(2) ou lors de la sauvegarde).

In [2]:


# Opening Spark session 
spark = SparkSession.builder.appName("INFO-H-515").master("local[*]").getOrCreate()

# Note : .master("local[*]") = Fais tourner en locale, et [*] = utiliser tous les cœurs de mon processeur pour paralléliser le travail
sc = spark.sparkContext

# Distribution between Spark workers
def extract_and_clean_pdf_with_spark(file_content):
    """
    Input : tuple (file_path, binary_content) thanks to sc.binaryFiles of Spark
    Return : liste of tuples : (file_name, page_number, cleaned_text).
    """
    file_path, file_binary = file_content
    file_name = file_path.split('/')[-1]
    pages_accumulator_data = []  # We have here the cleaned and filtered pages to output (ex : [ (FileA, Page1, "text..."), (FileA, Page2, "text...") ]
    try:
        # We already extracted files from memory and put them in bytes thanks to Spark.
        # We have to put them in a BytesIO object (closed book) to be able to read them with pdfreader.
        pdf_bytes_flux = io.BytesIO(file_binary)
        pdf_reader = PdfReader(pdf_bytes_flux)
        for page_number, page in enumerate(pdf_reader.pages):
            text = page.extract_text()
            if text:
                """
                Clean the text with project requirements: 
                line breaks,
                spurious line breaks,
                repeated whitespace,
                empty pages,
                headers/footers when they harm retrieval,
                and duplicated boilerplate
                """               
                text = re.sub(r'-\s*\n\s*', '', text) # Word break (ex: "distri- \n bution" -> "distribution").
                text = re.sub(r'(?m)^\s*\d+\s*$', '', text) # Delete page numbers (Conclusion after analyzing the extraction from pdf
                text = re.sub(r'(?<!\n)\n(?!\n)', ' ', text) # Line break but keep paragraphs 
                text = re.sub(r'\s+', ' ', text).strip() # Erase Multiple whitespaces
                
                if len(text) > 5: # Ignore empty pages or with very little text
                    pages_accumulator_data.append((file_name, page_number+ 1, text))

                # TODO : gerer les boilerplates, cad les element repetitif sans interet  ou plus value (analyser l'extraction des pdfs)

        return pages_accumulator_data

    except Exception as e:
        # Avoiding crash 
        print(f"Error with the document {file_name}: {str(e)}")
        return []

# input_dir = "C:/Users/kerim/Desktop/casquet/MA2/infoh515 BIG DATA/*.pdf" # Si non utilisation du container ( A enlever si pas besoin )
input_pdf_directory = "/home/guest/shared/course_material/*.pdf"
pdf_rdd= sc.binaryFiles(input_pdf_directory) # RDD Creation (Resilient Distributed Dataset) every element is a tuple (path, binary) 

"""
Note : diff entre map et flatmap 
map = [[PageA1, PageA2, PageA3], [PageB1, PageB2]] List de list = pas bon car que 2 processeurs vont travailler ici
flatMap = [PageA1, PageA2, PageA3, PageB1, PageB2] 
"""
cleaned_pages_in_rdd = pdf_rdd.flatMap(extract_and_clean_pdf_with_spark)
"""reminder for flatmap : only a receipe for when the code will need the output and (lazy evaluation) will run this function the number of times the code ask for it (in parallel)"""


print("PySpark verification :")
for page_info in cleaned_pages_in_rdd.take(3):
    print(page_info)

Using Spark's default log4j profile: org/apache/spark/log4j-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/14 20:40:18 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


PySpark verification :


('1-intro.pdf', 1, 'INFO-H515: BIG DATA: DISTRIBUTED MANAGEMENT Lecture 1: Introduction, HDFS, Map/Reduce Dimitris Sacharidis')
('1-intro.pdf', 2, 'LECTURE OUTLINE What is Big Data? Big Datacenters HDFS Map/Reduce')
('1-intro.pdf', 3, 'WHAT IS BIG DATA?')


In [3]:
# Analyse de l'extraction pour mieux visualiser ce qu'il faut nettoyer (A enlever pour le final ou pas ...)
# On cible un de vos fichiers précis
chemin = "/home/guest/shared/course_material/1-intro.pdf"
reader = PdfReader(chemin)

print("--- TEXTE BRUT DE LA PAGE 2 ---")
print(reader.pages[1].extract_text())

print("\n--- TEXTE BRUT DE LA PAGE 3 ---")
print(reader.pages[2].extract_text())

--- TEXTE BRUT DE LA PAGE 2 ---
LECTURE OUTLINE
What is Big Data?
Big Datacenters
HDFS
Map/Reduce
1

--- TEXTE BRUT DE LA PAGE 3 ---
WHAT IS BIG DATA?


In [4]:
def chunking_from_text(page_content, chunk_size = 300, overlap = 0.20):
    """
    Input : page_content : tuple (file_name, page_number, text) 
            chunk_size : number of words per chunk - default = 300 words
            overlap : overlap between chunks - default = 20%
    Output : list of tuples (file_name, page_number, chunk_id, start_word, end_word, chunk_text) where chunk_id is unique.
    """
    file_name, page_number, text = page_content
    list_of_word = text.split()     # Text separation in lists
    number_of_words = len(list_of_word)
    
    
    
    # Keep all if number_of_words <= chunk_size
    if number_of_words <= chunk_size:
        chunk_id = f"p{page_number}_chunk_index1"
        return [(file_name, page_number, chunk_id, 0, number_of_words, text)]
    
    chunks_accumulator = []
    overlap_words = int(chunk_size * overlap) # total words in overlap
    step = chunk_size - overlap_words 
    chunk_index = 1
      
      

    
    
    for chunk_first_word_index in range(0, number_of_words, step):
        chunk_last_word_index = min(chunk_first_word_index + chunk_size, number_of_words) # min is for dealing with the last chunk
        chunk = list_of_word[chunk_first_word_index:chunk_last_word_index] 
        chunk_text = " ".join(chunk) # extract chunk in form of list then join it
        chunk_id = f"p{page_number}_chunk_index{chunk_index}" # Making a unique id for each chunk (for better retrieval and traceability)
        chunks_accumulator.append((file_name,page_number,chunk_id,chunk_first_word_index,chunk_last_word_index,chunk_text)) # Add the chunk to the accumulator with all the metadata
        chunk_index += 1
        if chunk_last_word_index == number_of_words: # Handling last chunk (only "==" beacause chunklastwordindex is already min with number of words)
            break
            
            
    return chunks_accumulator









chunked_rdd = cleaned_pages_in_rdd.flatMap(lambda rdd: chunking_from_text(rdd, chunk_size=300, overlap=0.20)) 

print(chunked_rdd.take(3)) # Verification of metadata

[('1-intro.pdf', 1, 'p1_chunk_index1', 0, 12, 'INFO-H515: BIG DATA: DISTRIBUTED MANAGEMENT Lecture 1: Introduction, HDFS, Map/Reduce Dimitris Sacharidis'), ('1-intro.pdf', 2, 'p2_chunk_index1', 0, 10, 'LECTURE OUTLINE What is Big Data? Big Datacenters HDFS Map/Reduce'), ('1-intro.pdf', 3, 'p3_chunk_index1', 0, 4, 'WHAT IS BIG DATA?')]


In [5]:
def SBERT(data_pack):
    """c'est une ia qui vecotorise les text, Sbert n'est pas dans spark donc y a pas de .fit() ou .transform()"""
    
    import numpy as np
    from sentence_transformers import SentenceTransformer
    import uuid
    tuple_list = list(data_pack)
    if not tuple_list:
        return iter([])
    !pip install sentence-transformers
    from sentence_transformers import SentenceTransformer
    sbert_model = SentenceTransformer('all-MiniLM-L6-v2')

    texts = [elem[5] for elem in tuple_list] #retrieve the texts from input tuples 
    emb = sbert_model.encode(texts, normalize_embeddings=True) #normalize for the cosine similarity later, (emb is now a list of vectors)

    for index, chunk in enumerate(tuple_list):
        source_pdf, page_num, chunk_id, start_word, end_word, chunk_text = chunk
        embedding = emb[index].astype(np.float32).tolist()  # idk why but i have to put tolist() 

        yield (
            str(uuid.uuid4()), source_pdf, page_num, chunk_id, start_word, end_word, chunk_text,embedding,"SBERT") # yield is to transform it in a  generator  (to not keep a big stack with a return)




def Word2Vec(data_pack):
    """Word2Vec : c'est un dico de mot déjà connu"""
    tuple_list = list(data_pack)
    if not tuple_list:
        return iter([])
    import gensim.downloader as api
    w2v_model = api.load("glove-wiki-gigaword-50") # "word2vec-google-news-300" mais c'est plus lourd

    for chunk in tuple_list:
        source_pdf, page_num, chunk_id, start_word, end_word, chunk_text = chunk
        list_of_word = chunk_text.lower().split()
        vectors_of_words = []
        for word in list_of_word:
            if word in w2v_model:
                vectors_of_words.append(w2v_model[word])
        if vectors_of_words: # dans le cours: "generate chunk-level embeddings by averaging word vectors" 
            embedding_of_vector = np.mean(vectors_of_words, axis=0).astype(np.float32).tolist()
        else:
            embedding_of_vector = [0.0] * w2v_model.vector_size # no words in the chunk
        yield (
            str(uuid.uuid4()),source_pdf, page_num, chunk_id,start_word, end_word, chunk_text,embedding_of_vector,"Word2Vec")



# moule pour la sortie (c'est Spark qui exige ça)
schema = StructType([
    StructField("id", StringType(), False),
    StructField("source_pdf", StringType(), False),
    StructField("page_num", IntegerType(), False),
    StructField("chunk_id", StringType(), False),
    StructField("start_word", IntegerType(), False),
    StructField("end_word", IntegerType(), False),
    StructField("chunk_text", StringType(), False),
    StructField("embedding", ArrayType(FloatType()), False),
    StructField("embedding_strategy", StringType(), False)
])

embedding_strategy = "SBERT"  # ou "Word2Vec" ou "TF-IDF" ou "TOUS"
# embedding_strategy = input("Quelle embedding ? Attention à l'ortho et au majuscule(SBERT, Word2Vec, TF-IDF) : : ")

if embedding_strategy == "SBERT":
    rdd_of_embeddings = chunked_rdd.mapPartitions(SBERT) # mapPartitions isto apply the function at each datapack to not reload the model
    embeddings_dataframe = spark.createDataFrame(rdd_of_embeddings, schema)
    embeddings_dataframe.write.mode("overwrite").parquet("/home/guest/shared/course_material/embeddings_sbert.parquet") # il demande en format parquet dans le pdf 
    
    

# pareil que pour SBERT juste en haut mais avec Word2Vec
if embedding_strategy == "Word2Vec":
    rdd_of_embeddings = chunked_rdd.mapPartitions(Word2Vec)
    embeddings_dataframe = spark.createDataFrame(rdd_of_embeddings, schema)
    embeddings_dataframe.write.mode("overwrite").parquet("/home/guest/shared/course_material/embeddings_word2vec.parquet")





if embedding_strategy == "TF-IDF":
    # chunk_text → Tokenizer → words → HashingTF → tf_features → IDF → embedding
    schema = StructType([
        StructField("source_pdf", StringType(), False),
        StructField("page_num", IntegerType(), False),
        StructField("chunk_id", StringType(), False),
        StructField("start_word", IntegerType(), False),
        StructField("end_word", IntegerType(), False),
        StructField("chunk_text", StringType(), False)
    ])
    chunk_dataframe = spark.createDataFrame(chunked_rdd, schema)
    tokenizer = Tokenizer(inputCol="chunk_text", outputCol="words") 
    hashing = HashingTF(inputCol="words", outputCol="tf_features", numFeatures=10000)
    idf = IDF(inputCol="tf_features", outputCol="embedding")

    tokenized = tokenizer.transform(chunk_dataframe) # transforme les phrases en listes de mots et ajoute une colonne "words" avec ces listes de mots
    
    
    dataframe_with_tfFeature = hashing.transform(tokenized) # ajoute une colonne avec les TF 
    
    idf = idf.fit(dataframe_with_tfFeature)  # explication idf : Spark remarque que le mot "Le" est dans 100% des textes (il n'est donc pas important pour différencier les sujets). Par contre, le mot "Big Data" n'est que dans un seul texte. Spark se dit : "Ok, 'Big Data' est un mot très important, je vais lui donner un gros score."
    dataframe_with_tf_x_idf = idf.transform(dataframe_with_tfFeature)# ajoute une colonne avec les TF-IDF

    generate_unique_id = udf(lambda: str(uuid.uuid4()), StringType())
    
    dataframe_with_tf_x_idf_final = dataframe_with_tf_x_idf.withColumn("id", generate_unique_id()).withColumn("embedding_strategy", lit("TF-IDF"))
    dataframe_with_tf_x_idf_final = dataframe_with_tf_x_idf_final.select("id", "source_pdf", "page_num", "chunk_id", "start_word", "end_word", "chunk_text", "embedding", "embedding_strategy") # add missing columns and select the final ones

    dataframe_with_tf_x_idf_final.write.mode("overwrite").parquet("/home/guest/shared/course_material/embeddings_tfidf.parquet")


Traceback (most recent call last):
  File "/home/guest/spark/python/pyspark/serializers.py", line 437, in dumps
    return cloudpickle.dumps(obj, pickle_protocol)
  File "/home/guest/spark/python/pyspark/cloudpickle/cloudpickle_fast.py", line 73, in dumps
    cp.dump(obj)
  File "/home/guest/spark/python/pyspark/cloudpickle/cloudpickle_fast.py", line 563, in dump
    return Pickler.dump(self, obj)
TypeError: cannot pickle '_hashlib.HASH' object


PicklingError: Could not serialize object: TypeError: cannot pickle '_hashlib.HASH' object